# Sentiment Analysis of X (Twitter) Data
## Public Opinion Analysis Using Natural Language Processing

### Objective
This notebook analyzes sentiment expressed in X posts and categorizes content into **Positive, Negative, and Neutral** classes. It uses NLP preprocessing, statistical analysis, keyword analysis, and visualizations to understand patterns in public opinion.

### Dataset
- File: `X data.csv`
- Records: approximately 163,000
- Features:
  - `clean_text`: cleaned tweet/post text
  - `category`: sentiment label (`-1 = Negative`, `0 = Neutral`, `1 = Positive`)

> **Important limitation:** The dataset does not contain a timestamp/date column. Therefore, a genuine sentiment-over-time analysis cannot be performed without fabricating temporal information. This notebook instead focuses on sentiment distribution and topic/keyword patterns.

## Research Questions

1. What is the overall distribution of positive, negative, and neutral sentiments on X?
2. Which sentiment category dominates the political discussions in the dataset?
3. What words and topics occur most frequently in each sentiment category?
4. How does the vocabulary of positive and negative posts differ?
5. Which political entities or discussion topics are most prominent across sentiments?
6. What insights about public opinion can be inferred from the sentiment patterns?

In [ ]:
# Install packages if required
# !pip install pandas numpy matplotlib seaborn nltk wordcloud

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import re
from collections import Counter

sns.set_theme(style="whitegrid")
plt.rcParams["figure.figsize"] = (10, 6)

## 1. Load and Inspect the Dataset

In [ ]:
# Update the path if required
file_path = "X data.csv"

df = pd.read_csv(file_path)

print("Dataset shape:", df.shape)
print("\nColumns:", df.columns.tolist())
display(df.head())

In [ ]:
print("Dataset information:")
df.info()

print("\nMissing values:")
print(df.isnull().sum())

print("\nRaw sentiment label counts:")
print(df["category"].value_counts(dropna=False))

## 2. Data Cleaning and Sentiment Label Mapping

In [ ]:
# Remove rows with missing text or sentiment labels
df = df.dropna(subset=["clean_text", "category"]).copy()

# Convert numerical labels to integers
df["category"] = df["category"].astype(int)

# Map sentiment labels to readable categories
sentiment_map = {
    -1: "Negative",
     0: "Neutral",
     1: "Positive"
}

df["sentiment"] = df["category"].map(sentiment_map)

print("Cleaned dataset shape:", df.shape)
print("\nSentiment distribution:")
print(df["sentiment"].value_counts())
display(df.head())

## 3. Statistical Sentiment Analysis

In [ ]:
sentiment_counts = df["sentiment"].value_counts().reindex(
    ["Positive", "Neutral", "Negative"]
)

sentiment_percentages = (
    df["sentiment"]
    .value_counts(normalize=True)
    .reindex(["Positive", "Neutral", "Negative"]) * 100
)

summary = pd.DataFrame({
    "Count": sentiment_counts,
    "Percentage": sentiment_percentages.round(2)
})

display(summary)

print("\nTotal posts analyzed:", len(df))
print("Dominant sentiment:", sentiment_counts.idxmax())

## 4. Visualization: Sentiment Distribution Bar Chart

In [ ]:
plt.figure(figsize=(9, 6))
ax = sns.countplot(
    data=df,
    x="sentiment",
    order=["Positive", "Neutral", "Negative"]
)

plt.title("Distribution of Sentiments in X Posts", fontsize=16, fontweight="bold")
plt.xlabel("Sentiment")
plt.ylabel("Number of Posts")

for container in ax.containers:
    ax.bar_label(container, fmt="%d")

plt.tight_layout()
plt.show()

## 5. Visualization: Sentiment Percentage Pie Chart

In [ ]:
plt.figure(figsize=(8, 8))

plt.pie(
    sentiment_percentages,
    labels=sentiment_percentages.index,
    autopct="%1.1f%%",
    startangle=90,
    explode=[0.03, 0.03, 0.03]
)

plt.title("Sentiment Composition of X Discussions", fontsize=16, fontweight="bold")
plt.show()

## 6. NLP Text Processing

In [ ]:
def preprocess_text(text):
    text = str(text).lower()
    text = re.sub(r"http\S+|www\S+", "", text)       # URLs
    text = re.sub(r"@\w+", "", text)                  # Mentions
    text = re.sub(r"#", "", text)                      # Remove hashtag symbol
    text = re.sub(r"[^a-z\s]", " ", text)             # Keep alphabetic text
    text = re.sub(r"\s+", " ", text).strip()          # Extra spaces
    return text

df["processed_text"] = df["clean_text"].apply(preprocess_text)

display(df[["clean_text", "processed_text", "sentiment"]].head())

## 7. Most Frequent Words by Sentiment

This analysis identifies the most commonly occurring words within Positive, Negative, and Neutral posts. Common stopwords are removed to focus on meaningful discussion terms.

In [ ]:
stop_words = {
    "the", "and", "for", "that", "this", "with", "you", "are", "was",
    "have", "not", "but", "all", "from", "they", "his", "her", "she",
    "him", "our", "your", "about", "just", "what", "when", "who", "how",
    "why", "will", "would", "should", "could", "into", "out", "their",
    "them", "been", "were", "had", "has", "its", "im", "is", "it",
    "to", "of", "in", "on", "a", "an", "at", "as", "be", "by", "or",
    "if", "we", "i", "me", "my", "do", "does", "did", "so", "than",
    "then", "more", "most", "very", "can", "cant", "dont", "rt", "amp",
    "via", "only", "one", "now", "like"
}

def get_top_words(text_series, n=20):
    text = " ".join(text_series.astype(str))
    words = re.findall(r"\b[a-z]{3,}\b", text.lower())
    words = [word for word in words if word not in stop_words]
    return Counter(words).most_common(n)

top_words = {}

for sentiment in ["Positive", "Negative", "Neutral"]:
    top_words[sentiment] = get_top_words(
        df.loc[df["sentiment"] == sentiment, "processed_text"]
    )
    print(f"\nTop words in {sentiment} posts:")
    print(top_words[sentiment])

## 8. Visualization: Top 15 Words in Each Sentiment Category

In [ ]:
fig, axes = plt.subplots(3, 1, figsize=(12, 18))

for ax, sentiment in zip(axes, ["Positive", "Negative", "Neutral"]):
    words, counts = zip(*top_words[sentiment][:15])

    sns.barplot(x=list(counts), y=list(words), ax=ax)
    ax.set_title(f"Top 15 Words in {sentiment} Posts", fontsize=14, fontweight="bold")
    ax.set_xlabel("Frequency")
    ax.set_ylabel("Words")

plt.tight_layout()
plt.show()

## 9. Word Clouds

In [ ]:
# Uncomment the installation line if WordCloud is not installed
# !pip install wordcloud

from wordcloud import WordCloud

fig, axes = plt.subplots(1, 3, figsize=(20, 7))

for ax, sentiment in zip(axes, ["Positive", "Negative", "Neutral"]):
    text = " ".join(
        df.loc[df["sentiment"] == sentiment, "processed_text"].astype(str)
    )

    wordcloud = WordCloud(
        width=800,
        height=500,
        background_color="white",
        stopwords=stop_words,
        max_words=100
    ).generate(text)

    ax.imshow(wordcloud, interpolation="bilinear")
    ax.set_title(f"{sentiment} Sentiment Word Cloud", fontsize=14, fontweight="bold")
    ax.axis("off")

plt.tight_layout()
plt.show()

## 10. Sentiment Comparison for Key Political Terms

In [ ]:
# Terms are selected based on recurring discussion topics in the dataset
keywords = ["modi", "india", "bjp", "congress", "government", "election", "vote", "rahul"]

keyword_results = []

for keyword in keywords:
    for sentiment in ["Positive", "Neutral", "Negative"]:
        count = df[
            (df["sentiment"] == sentiment) &
            (df["processed_text"].str.contains(rf"\b{keyword}\b", regex=True, na=False))
        ].shape[0]

        keyword_results.append({
            "Keyword": keyword,
            "Sentiment": sentiment,
            "Count": count
        })

keyword_df = pd.DataFrame(keyword_results)
display(keyword_df.pivot(index="Keyword", columns="Sentiment", values="Count"))

## 11. Visualization: Keyword Mentions Across Sentiments

In [ ]:
pivot_keywords = keyword_df.pivot(
    index="Keyword",
    columns="Sentiment",
    values="Count"
).fillna(0)

pivot_keywords = pivot_keywords.reindex(
    columns=["Positive", "Neutral", "Negative"]
)

pivot_keywords.plot(
    kind="bar",
    figsize=(14, 7)
)

plt.title("Sentiment Distribution Across Major Discussion Keywords", fontsize=16, fontweight="bold")
plt.xlabel("Keyword")
plt.ylabel("Number of Posts")
plt.xticks(rotation=0)
plt.legend(title="Sentiment")
plt.tight_layout()
plt.show()

## 12. Tweet/Post Length Analysis

In [ ]:
df["text_length"] = df["processed_text"].str.split().str.len()

length_summary = df.groupby("sentiment")["text_length"].describe()
display(length_summary)

In [ ]:
plt.figure(figsize=(10, 6))

sns.boxplot(
    data=df,
    x="sentiment",
    y="text_length",
    order=["Positive", "Neutral", "Negative"],
    showfliers=False
)

plt.title("Post Length Distribution by Sentiment", fontsize=16, fontweight="bold")
plt.xlabel("Sentiment")
plt.ylabel("Number of Words")
plt.tight_layout()
plt.show()

## 13. Statistical Test: Chi-Square Goodness of Fit

In [ ]:
from scipy.stats import chisquare

observed = sentiment_counts.values
expected = np.repeat(observed.mean(), len(observed))

chi2, p_value = chisquare(f_obs=observed, f_exp=expected)

print("Chi-square statistic:", round(chi2, 2))
print("P-value:", p_value)

if p_value < 0.05:
    print("\nResult: The sentiment distribution is significantly different from an equal distribution.")
else:
    print("\nResult: No statistically significant difference from an equal distribution was detected.")

# 14. Key Findings and Insights

The results should be interpreted after running the notebook. Based on the dataset structure and computed analysis, focus on the following:

1. **Overall sentiment:** Identify whether Positive, Neutral, or Negative sentiment dominates.
2. **Public discourse:** Compare the proportion of positive and negative discussions to understand the general tone.
3. **Topic patterns:** Examine the most frequent words and keyword-level sentiment differences.
4. **Political discussion:** Terms such as political leaders, parties, elections, governance, and India may appear frequently across sentiment categories.
5. **Vocabulary differences:** Positive and negative posts may use different descriptive terms even when discussing the same political entities.
6. **Statistical significance:** The chi-square test evaluates whether the observed sentiment distribution differs significantly from an equal distribution.
7. **Limitation:** Since the dataset does not contain timestamps, no valid sentiment-over-time trend should be claimed.

# 15. Conclusion

This analysis demonstrates how Natural Language Processing can be used to understand public sentiment expressed on X. The dataset was categorized into Positive, Neutral, and Negative sentiment classes and analyzed using descriptive statistics, word-frequency analysis, keyword comparisons, and visualizations.

The combination of sentiment distribution charts, word clouds, frequent-word analysis, and topic-level comparisons provides a broader understanding of the discussions represented in the dataset. The findings can support research into public opinion, political discourse, and online social-media behavior.

## Future Scope

If timestamps and additional metadata become available, future analysis can include:
- Sentiment trends over time
- Event-based sentiment spikes
- Topic modeling using LDA
- Named Entity Recognition
- Emotion classification
- Geographic sentiment analysis
- Transformer-based sentiment validation using BERT or RoBERTa